# Notebook 01 — Data pipeline

**Phase 1 task.** Stand up data ingestion from two sources:
1. **Binance REST API** — free, no account needed, tick-level trades + order book
2. **LOBSTER** — NASDAQ limit order book data (academic free tier)

By the end of this notebook you should have:
- A parquet file of Binance BTCUSDT trades at 1-second resolution in `data/processed/`
- Functions you can import in later notebooks to fetch fresh data

**Do not commit raw data files to git.** `data/raw/` is gitignored.

In [ ]:
import requests
import pandas as pd
import numpy as np
from pathlib import Path
import time

DATA_RAW = Path('../data/raw')
DATA_PROC = Path('../data/processed')
DATA_RAW.mkdir(exist_ok=True)
DATA_PROC.mkdir(exist_ok=True)

## 1. Binance REST API

Binance's public API requires no API key for historical data. Rate limit: 1200 requests/min.

Useful endpoints:
- `/api/v3/aggTrades` — aggregated trades (timestamp, price, qty, buyer is maker)
- `/api/v3/depth` — current order book snapshot
- `/api/v3/klines` — OHLCV candlesticks

**TODO:** Complete the `fetch_binance_trades()` function below.

In [ ]:
BINANCE_BASE = 'https://api.binance.com'

def fetch_binance_trades(symbol: str = 'BTCUSDT', limit: int = 1000) -> pd.DataFrame:
    """
    Fetch the most recent `limit` aggregated trades for `symbol`.
    
    Returns a DataFrame with columns:
        timestamp (datetime), price (float), qty (float), side (+1 buy / -1 sell)
    
    TODO: extend to fetch historical data by paginating with `fromId`.
    Binance only returns 1000 trades per call, so for a full day of data
    you need to page through ~50-100 calls with a small sleep between them.
    """
    url = f'{BINANCE_BASE}/api/v3/aggTrades'
    params = {'symbol': symbol, 'limit': limit}
    
    resp = requests.get(url, params=params, timeout=10)
    resp.raise_for_status()
    raw = resp.json()
    
    df = pd.DataFrame(raw)
    df = df.rename(columns={
        'T': 'timestamp',
        'p': 'price',
        'q': 'qty',
        'm': 'buyer_is_maker',
    })[['timestamp', 'price', 'qty', 'buyer_is_maker']]
    
    df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms')
    df['price'] = df['price'].astype(float)
    df['qty'] = df['qty'].astype(float)
    # buyer_is_maker=True means the buyer was passive → seller-initiated trade
    df['side'] = df['buyer_is_maker'].map({True: -1, False: 1})
    df = df.drop(columns='buyer_is_maker')
    
    return df

# Quick test
trades = fetch_binance_trades()
print(f'Fetched {len(trades)} trades')
trades.head()

## 2. Save to parquet

Parquet is the right format here: columnar, compressed, fast to load in pandas/polars.
Avoid CSV for tick data — it's 5-10x larger and slower to parse.

In [ ]:
out_path = DATA_PROC / 'binance_btcusdt_trades_sample.parquet'
trades.to_parquet(out_path, index=False)
print(f'Saved to {out_path} ({out_path.stat().st_size / 1024:.1f} KB)')

## 3. LOBSTER data (academic)

**TODO:** Register at lobsterdata.com for a free academic account.
Download 1 day of AAPL or MSFT level-5 LOB data.
LOBSTER provides two files per day:
- `*_message_*.csv` — order events (timestamps, event type, price, size, direction)
- `*_orderbook_*.csv` — LOB snapshots after each event

Add a `load_lobster()` function below that reads both files and merges them.
Reference: LOBSTER documentation at https://lobsterdata.com/info/DataStructure.php

In [ ]:
def load_lobster(message_file: str, orderbook_file: str, levels: int = 5) -> pd.DataFrame:
    """
    Load and merge LOBSTER message + orderbook files.
    
    TODO: implement this in Phase 2.
    See LOBSTER data structure docs for column specifications.
    """
    raise NotImplementedError(
        'load_lobster() is a Phase 2 task. '
        'See LOBSTER docs: https://lobsterdata.com/info/DataStructure.php'
    )